In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

from transformer import ThermalTransformer
from loader import load_data

In [2]:
holdout = None
#holdout = [-12.502,-29.500,-41.010]

train_loader,train_val_loader,test_loader = load_data('../data/full_dataset/data_rearranged.csv',holdout=holdout,batch_size=16)

In [3]:
base_lr = 1e-4
thermal_lr = 8e-2
hidden_dim = 200

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ThermalTransformer(input_dim=5,d_model=hidden_dim,nhead=5).to(device).float()
criterion = nn.MSELoss()

thermal_params = [p for n, p in model.named_parameters() if 'log_beta' in n or 'mu' in n]
base_params = [p for n, p in model.named_parameters() if 'log_beta' not in n and 'mu' not in n]

optimizer = torch.optim.Adam([
    {'params': base_params, 'lr': base_lr}, 
    {'params': thermal_params, 'lr': thermal_lr}
])

num_epochs = 100

print(f"Training on {device}...")

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device).float(), batch_y.to(device).float()
        
        preds = model(batch_x)
        mse_loss = criterion(preds, batch_y)
        rmse_loss = torch.sqrt(mse_loss + 1e-15)
        
        optimizer.zero_grad()
        rmse_loss.backward()
        optimizer.step()
        
        total_train_loss += rmse_loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    model.eval()
    total_test_loss = 0
    
    with torch.no_grad():
        for batch_x, batch_y in train_val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            preds = model(batch_x)
            mse_loss = criterion(preds, batch_y)
            rmse_loss = torch.sqrt(mse_loss + 1e-15)
            total_test_loss += rmse_loss.item()
            
    avg_test_loss = total_test_loss / len(test_loader)
    
    # --- LOGGING ---
    if (epoch + 1) % 5 == 0 or epoch == 0:
        sample_beta = np.exp([x.log_beta.item() for x in model.heads])
        sample_mu = np.exp([x.mu.item() for x in model.heads])
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train RMSE: {avg_train_loss:.4f} | Test RMSE: {avg_test_loss:.4f} | Beta: {sample_beta} | Mu: {sample_mu}")
        print("-" * 30)

model.eval()
total_test_loss = 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        preds = model(batch_x)
        mse_loss = criterion(preds, batch_y)
        rmse_loss = torch.sqrt(mse_loss + 1e-15)
        total_test_loss += rmse_loss.item()
        
avg_test_loss = total_test_loss / len(test_loader)
print(f"Final Holdout Validation RMSE: {avg_test_loss}")

Training on cuda...
Epoch [1/100]
  Train RMSE: 5.4940 | Test RMSE: 0.9023 | Beta: [48.53353864 28.58452414 10.95848734 20.4903533   0.70152705] | Mu: [0.40096658 1.45032703 1.53514431 1.36031762 0.66727428]
------------------------------
Epoch [5/100]
  Train RMSE: 1.6585 | Test RMSE: 1.0530 | Beta: [54.36604023 28.58583267 28.2497666  16.7687117   1.39995445] | Mu: [0.34654018 1.45023708 1.53526057 1.36045243 0.10076977]
------------------------------
Epoch [10/100]
  Train RMSE: 1.4948 | Test RMSE: 0.6126 | Beta: [69.69537427 28.58979269 13.19272029 10.89469378  2.47560257] | Mu: [0.26574798 1.44994028 1.53521779 1.36042992 0.43473628]
------------------------------
Epoch [15/100]
  Train RMSE: 1.3796 | Test RMSE: 0.4967 | Beta: [69.69417788 28.59853939  9.70678612  7.7684045   3.22415566] | Mu: [0.26575862 1.44925091 1.53529036 1.3605055  0.08723843]
------------------------------
Epoch [20/100]
  Train RMSE: 1.1826 | Test RMSE: 0.4306 | Beta: [73.24460883 28.59780301 11.23459396  

In [11]:
# Specify a path
from datetime import datetime

now = datetime.now()
formatted_time = now.strftime("%d%B%Y-%H%M")
out_path = f"thermal_transformer_hidden-{hidden_dim}_base-{base_lr}_thermal-{thermal_lr}_{formatted_time}.pth"

torch.save(model.state_dict(), out_path)
print(f"Model parameters saved to {out_path}")

Model parameters saved to thermal_transformer_hidden-200_base-0.0001_thermal-0.08_26April2026-2332.pth


In [ ]:
model = ThermalTransformer(input_dim=5, d_model=64, nhead=8)
model.load_state_dict(torch.load("thermal_transformer_26April2026-2150.pth"))
model.eval()

total_test_loss = 0

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        preds = model(batch_x)
        mse_loss = criterion(preds, batch_y)
        rmse_loss = torch.sqrt(mse_loss + 1e-15)
        total_test_loss += rmse_loss.item()
        
avg_test_loss = total_test_loss / len(test_loader)
print(f"Final MSE: {avg_test_loss}")